# Phase 1 -- Medium Starting Kit (Competition-Grade)


This notebook implements a blend of **CatBoost quantile** and **LGBM**  for each horizon.

**Prerequisites**: Run `1_feature_engineering.ipynb` first.

**Runtime**: ~30-60 minutes .


## 1. Setup

In [1]:
import os, sys
# Ensure we're in the notebook's directory (participant_kit/phase1/)
_this_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else None
if _this_dir is None:
    for _candidate in [os.getcwd(), os.path.join(os.getcwd(), "participant_kit", "phase1")]:
        if os.path.exists(os.path.join(_candidate, "utils.py")):
            os.chdir(_candidate)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import warnings
from pathlib import Path
from time import time

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)

from utils import (
    REGIONS, HORIZONS, HOURS, QUANTILES_DEFAULT, TOP_K_DEFAULT,
    WORLDWIDE_PREFIXES,
    load_train_data, load_inference_features, load_vertical_ratios,
    exclude_worldwide_features, load_or_compute_selection,
    winkler_score, circular_mae,
    predict_speed_10m, predict_direction, merge_speed_direction,
    expand_to_all_levels, generate_submission,
)

# -- Paths --
DATA_DIR = Path("phase1_dataset")
FEATURES_DIR = DATA_DIR / "features"

assert FEATURES_DIR.exists(), (
    f"Features directory not found: {FEATURES_DIR}\n"
    "Run 1_feature_engineering.ipynb first!"
)

# -- Multi-seed configuration --
SEEDS = [42, 123, 456, 789, 2024]
MAX_TRAIN_SAMPLES = 500_000

print(f"Features directory: {FEATURES_DIR.resolve()}")
print(f"Regions: {REGIONS}")
print(f"Seeds for CatBoost ensembling: {SEEDS}")
print(f"Max training samples: {MAX_TRAIN_SAMPLES:,}")


Features directory: /Users/david/projects/hackathon_2026/wind-prediction-private/participant_kit/phase1/phase1_dataset/features
Regions: ['north_sea', 'east_china_sea']
Seeds for CatBoost ensembling: [42, 123, 456, 789, 2024]
Max training samples: 500,000


## 2. Load Data

Load pre-computed training features for both regions. Exclude worldwide features
(NAO, Siberian High, etc.) for speed models -- they add noise without helping
short/medium-range wind speed prediction.

In [2]:
# Load training data for both regions
reanalysis_feat_all = {}
feature_cols_all = {}   # all features (for direction models)
speed_fcols_all = {}    # filtered features (for speed models, no worldwide)
speed_targets = None
dir_targets = None

for region in REGIONS:
    df, fcols, s_tgts, d_tgts = load_train_data(FEATURES_DIR, region)
    reanalysis_feat_all[region] = df
    feature_cols_all[region] = fcols
    speed_fcols_all[region] = exclude_worldwide_features(fcols)

    if speed_targets is None:
        speed_targets = s_tgts
        dir_targets = d_tgts

    print(f"\n{region}:")
    print(f"  Shape: {df.shape}")
    print(f"  Time range: {df['time'].min()} to {df['time'].max()}")
    print(f"  All features: {len(fcols)}")
    print(f"  Speed features (no worldwide): {len(speed_fcols_all[region])}")
    print(f"  Grid points: {df.groupby(['latitude', 'longitude']).ngroups}")

print(f"\nSpeed targets ({len(speed_targets)}): {speed_targets}")
print(f"Direction targets ({len(dir_targets)}): {dir_targets}")



north_sea:
  Shape: (2811240, 150)
  Time range: 2019-01-01 00:00:00 to 2021-12-31 00:00:00
  All features: 125
  Speed features (no worldwide): 74
  Grid points: 2565



east_china_sea:
  Shape: (2811240, 150)
  Time range: 2019-01-01 00:00:00 to 2021-12-31 00:00:00
  All features: 125
  Speed features (no worldwide): 74
  Grid points: 2565

Speed targets (12): ['speed_d14_h0', 'speed_d14_h12', 'speed_d14_h18', 'speed_d14_h6', 'speed_d1_h0', 'speed_d1_h12', 'speed_d1_h18', 'speed_d1_h6', 'speed_d7_h0', 'speed_d7_h12', 'speed_d7_h18', 'speed_d7_h6']
Direction targets (12): ['dir_d14_h0', 'dir_d14_h12', 'dir_d14_h18', 'dir_d14_h6', 'dir_d1_h0', 'dir_d1_h12', 'dir_d1_h18', 'dir_d1_h6', 'dir_d7_h0', 'dir_d7_h12', 'dir_d7_h18', 'dir_d7_h6']


## 3. Feature Selection

Per-target CatBoost-based feature importance. Results are cached to JSON so
re-running the notebook skips this step. Delete the cache file or set `force=True`
to recompute.

In [3]:
selected_features_all = {}  # {region: {target: [feature_list]}}

for region in REGIONS:
    cache_path = FEATURES_DIR / f"feature_selection_catboost_{region}.json"
    selected_features_all[region] = load_or_compute_selection(
        cache_path=cache_path,
        df=reanalysis_feat_all[region],
        feature_cols=speed_fcols_all[region],
        speed_targets=speed_targets,
        model_type="catboost",
        top_k=TOP_K_DEFAULT,
        force=False,
    )
    # Print summary
    for tgt in speed_targets[:3]:
        feats = selected_features_all[region][tgt]
        print(f"  {region}/{tgt}: {len(feats)} features -> {feats[:5]}...")
    print()


Loaded cached feature selection: feature_selection_catboost_north_sea.json (12 targets)
  north_sea/speed_d14_h0: 25 features -> ['sst', 'woy_cos', 'woy_sin', 'latitude', 'msl_lag7d']...
  north_sea/speed_d14_h12: 25 features -> ['sst', 'woy_cos', 'woy_sin', 'latitude', 'msl_lag7d']...
  north_sea/speed_d14_h18: 25 features -> ['sst', 'woy_cos', 'woy_sin', 'latitude', 'elevation']...

Loaded cached feature selection: feature_selection_catboost_east_china_sea.json (12 targets)
  east_china_sea/speed_d14_h0: 25 features -> ['sst', 'woy_cos', 'woy_sin', 'ws10_rmean7d', 'latitude']...
  east_china_sea/speed_d14_h12: 25 features -> ['sst', 'woy_cos', 'woy_sin', 'ws10_rmean7d', 'latitude']...
  east_china_sea/speed_d14_h18: 25 features -> ['sst', 'woy_cos', 'woy_sin', 'ws10_rmean7d', 'elevation']...



## 4. Train/Val Split

Train on 2019-2020, validate on full 2021. Subsample to 500K rows per region
to keep training times manageable.

In [4]:
df_train_all = {}
df_val_all = {}

for region in REGIONS:
    df = reanalysis_feat_all[region]
    valid_mask = df[speed_targets[0]].notna()

    # Training: 2019-2020
    train_mask = df["time"].dt.year.isin([2019, 2020])
    df_full = df[train_mask & valid_mask].copy()
    if len(df_full) > MAX_TRAIN_SAMPLES:
        df_train_all[region] = df_full.sample(n=MAX_TRAIN_SAMPLES, random_state=42)
        print(f"  {region}: subsampled {len(df_full):,} -> {MAX_TRAIN_SAMPLES:,}")
    else:
        df_train_all[region] = df_full

    # Validation: full 2021
    val_mask = df["time"].dt.year == 2021
    df_val_all[region] = df[val_mask & valid_mask].copy()

    print(f"  {region}: train={len(df_train_all[region]):,}, "
          f"val={len(df_val_all[region]):,} (full 2021)")

  north_sea: subsampled 1,875,015 -> 500,000
  north_sea: train=500,000, val=156,465 (Nov-Dec 2021)


  east_china_sea: subsampled 1,875,015 -> 500,000
  east_china_sea: train=500,000, val=156,465 (Nov-Dec 2021)


## 5. Asymmetric Quantile Search

Here slightly asymmetric intervals (e.g., 0.04/0.96) yield
lower Winkler scores, because real wind speed error distributions are right-skewed.

In [5]:
LO_ALPHA = 0.04
HI_ALPHA = 0.96
QUANTILES = [LO_ALPHA, 0.5, HI_ALPHA]  # Used throughout the notebook
print(f"Using pre-tuned quantile alphas: lo={LO_ALPHA}, hi={HI_ALPHA}")
print(f"Quantiles: {QUANTILES}")


Using pre-tuned quantile alphas: lo=0.04, hi=0.96
Quantiles: [0.04, 0.5, 0.96]


## 6. CatBoost Speed Models -- Multi-Seed

Train 5 CatBoost models per quantile per target (one per seed), then average
predictions at inference time.

In [6]:
q_lo, q_mid, q_hi = QUANTILES

def train_catboost_seed(X_train, y_train, quantile, feats, seed):
    """Train a single CatBoost quantile model with HRES golden borders."""
    params = dict(
        loss_function=f"Quantile:alpha={quantile}",
        iterations=500,
        depth=7,
        learning_rate=0.05,
        l2_leaf_reg=3,
        random_seed=seed,
        verbose=0,
    )
    # HRES features get higher quantization resolution (1024 vs default 254)
    hres_indices = [i for i, f in enumerate(feats) if f.startswith("fcst_speed")]
    if hres_indices:
        params["per_float_feature_quantization"] = [
            f"{i}:border_count=1024" for i in hres_indices
        ]
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train)
    return model


# Train all speed models: {region: {target: [list_of_seed_model_dicts]}}
cb_speed_models = {}
cb_val_results = {}

t0_total = time()

for region in REGIONS:
    print(f"\n{'='*65}")
    print(f"CatBoost multi-seed training: {region}")
    print(f"{'='*65}")
    df_tr = df_train_all[region]
    df_vl = df_val_all[region]

    cb_speed_models[region] = {}
    region_results = []

    for tgt in speed_targets:
        mask_tr = df_tr[tgt].notna()
        mask_vl = df_vl[tgt].notna()
        if mask_tr.sum() < 100 or mask_vl.sum() < 100:
            continue

        feats = selected_features_all[region][tgt]
        X_tr = df_tr.loc[mask_tr, feats].fillna(0)
        y_tr = df_tr.loc[mask_tr, tgt]
        X_vl = df_vl.loc[mask_vl, feats].fillna(0)
        y_vl = df_vl.loc[mask_vl, tgt].values

        t0 = time()
        seed_models = []
        for seed in SEEDS:
            m_dict = {}
            for q in QUANTILES:
                m_dict[q] = train_catboost_seed(X_tr, y_tr, q, feats, seed)
            seed_models.append(m_dict)

        cb_speed_models[region][tgt] = seed_models
        elapsed = time() - t0

        # Evaluate: average across seeds
        q05_preds = np.mean([
            np.maximum(sm[q_lo].predict(X_vl), 0) for sm in seed_models
        ], axis=0)
        q50_preds = np.mean([sm[q_mid].predict(X_vl) for sm in seed_models], axis=0)
        q95_preds = np.mean([sm[q_hi].predict(X_vl) for sm in seed_models], axis=0)

        alpha_width = 1.0 - (q_hi - q_lo)
        ws = winkler_score(y_vl, q05_preds, q95_preds, alpha=alpha_width)
        cov = np.mean((y_vl >= q05_preds) & (y_vl <= q95_preds)) * 100
        rmse = np.sqrt(np.nanmean((y_vl - q50_preds) ** 2))

        horizon = int(tgt.split("_")[1][1:])
        hour = int(tgt.split("_")[2][1:])
        region_results.append({
            "target": tgt, "horizon": horizon, "hour": hour,
            "ws": ws, "coverage": cov, "rmse": rmse,
        })
        print(f"  {tgt}: WS={ws:.2f}, cov={cov:.0f}%, RMSE={rmse:.2f} "
              f"({len(SEEDS)} seeds, {elapsed:.0f}s)")

    cb_val_results[region] = pd.DataFrame(region_results)
    print(f"\n{region} -- CatBoost mean by horizon:")
    print(cb_val_results[region].groupby("horizon")[["ws", "coverage", "rmse"]]
          .mean().round(2))

print(f"\nTotal CatBoost training time: {time() - t0_total:.0f}s")


CatBoost multi-seed training: north_sea


  speed_d14_h0: WS=14.97, cov=80%, RMSE=3.39 (5 seeds, 133s)


  speed_d14_h12: WS=15.43, cov=79%, RMSE=3.52 (5 seeds, 135s)


  speed_d14_h18: WS=15.69, cov=79%, RMSE=3.49 (5 seeds, 137s)


  speed_d14_h6: WS=15.24, cov=78%, RMSE=3.47 (5 seeds, 137s)


...

  speed_d1_h6: WS=4.23, cov=91%, RMSE=0.99 (5 seeds, 112s)


  speed_d7_h0: WS=9.70, cov=86%, RMSE=2.12 (5 seeds, 115s)


  speed_d7_h12: WS=10.13, cov=86%, RMSE=2.30 (5 seeds, 116s)


  speed_d7_h18: WS=10.21, cov=84%, RMSE=2.37 (5 seeds, 119s)


  speed_d7_h6: WS=10.34, cov=84%, RMSE=2.28 (5 seeds, 114s)

east_china_sea -- CatBoost mean by horizon:
            ws  coverage  rmse
horizon                       
1         4.38     91.21  1.03
7        10.10     84.87  2.27
14       13.18     81.33  2.89

Total CatBoost training time: 3053s


## 7. LGBM Tuned -- For Blending

Train a complementary set of LGBM models with more trees, lower learning rate, and
early stopping.

In [7]:
import lightgbm as lgb

def train_lgbm_tuned(X_train, y_train, X_val, y_val, quantile=0.5):
    """Train a tuned LGBM quantile model with early stopping."""
    params = {
        "objective": "quantile", "alpha": quantile, "metric": "quantile",
        "n_estimators": 2000, "max_depth": 7, "learning_rate": 0.01,
        "subsample": 0.8, "colsample_bytree": 0.8, "min_child_samples": 50,
        "num_leaves": 63, "verbose": -1, "n_jobs": -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    return model


# Train LGBM for blending: {region: {target: {quantile: model}}}
lgbm_blend_models = {}
lgbm_val_results = {}

t0_total = time()

for region in REGIONS:
    print(f"\n{'='*65}")
    print(f"LGBM tuned training: {region}")
    print(f"{'='*65}")
    df_tr = df_train_all[region]
    df_vl = df_val_all[region]

    lgbm_blend_models[region] = {}
    region_results = []

    for tgt in speed_targets:
        mask_tr = df_tr[tgt].notna()
        mask_vl = df_vl[tgt].notna()
        if mask_tr.sum() < 100 or mask_vl.sum() < 100:
            continue

        feats = selected_features_all[region][tgt]
        X_tr = df_tr.loc[mask_tr, feats].fillna(0)
        y_tr = df_tr.loc[mask_tr, tgt]
        X_vl = df_vl.loc[mask_vl, feats].fillna(0)
        y_vl = df_vl.loc[mask_vl, tgt]

        t0 = time()
        models = {}
        for q in QUANTILES:
            models[q] = train_lgbm_tuned(X_tr, y_tr, X_vl, y_vl.values, quantile=q)
        lgbm_blend_models[region][tgt] = models
        elapsed = time() - t0

        # Evaluate
        q05_p = np.maximum(models[q_lo].predict(X_vl), 0)
        q50_p = models[q_mid].predict(X_vl)
        q95_p = models[q_hi].predict(X_vl)

        alpha_width = 1.0 - (q_hi - q_lo)
        ws = winkler_score(y_vl.values, q05_p, q95_p, alpha=alpha_width)
        cov = np.mean((y_vl.values >= q05_p) & (y_vl.values <= q95_p)) * 100
        rmse = np.sqrt(np.nanmean((y_vl.values - q50_p) ** 2))

        n_trees = max(models[q].best_iteration_ or models[q].n_estimators
                      for q in QUANTILES)
        horizon = int(tgt.split("_")[1][1:])
        hour = int(tgt.split("_")[2][1:])
        region_results.append({
            "target": tgt, "horizon": horizon, "hour": hour,
            "ws": ws, "coverage": cov, "rmse": rmse,
        })
        print(f"  {tgt}: WS={ws:.2f}, cov={cov:.0f}%, RMSE={rmse:.2f} "
              f"(best_iter~{n_trees}, {elapsed:.0f}s)")

    lgbm_val_results[region] = pd.DataFrame(region_results)
    print(f"\n{region} -- LGBM mean by horizon:")
    print(lgbm_val_results[region].groupby("horizon")[["ws", "coverage", "rmse"]]
          .mean().round(2))

print(f"\nTotal LGBM training time: {time() - t0_total:.0f}s")


LGBM tuned training: north_sea


  speed_d14_h0: WS=13.19, cov=90%, RMSE=3.36 (best_iter~291, 27s)


  speed_d14_h12: WS=13.43, cov=89%, RMSE=3.48 (best_iter~325, 29s)


  speed_d14_h18: WS=13.69, cov=89%, RMSE=3.41 (best_iter~214, 23s)


  speed_d14_h6: WS=13.08, cov=89%, RMSE=3.40 (best_iter~370, 33s)


...

  speed_d1_h6: WS=4.26, cov=91%, RMSE=0.99 (best_iter~1779, 151s)


  speed_d7_h0: WS=9.07, cov=91%, RMSE=2.13 (best_iter~467, 46s)


  speed_d7_h12: WS=9.62, cov=89%, RMSE=2.28 (best_iter~440, 45s)


  speed_d7_h18: WS=9.51, cov=89%, RMSE=2.34 (best_iter~450, 41s)


  speed_d7_h6: WS=9.66, cov=89%, RMSE=2.27 (best_iter~530, 48s)

east_china_sea -- LGBM mean by horizon:
            ws  coverage  rmse
horizon                       
1         4.41     91.05  1.02
7         9.47     89.41  2.25
14       11.81     88.59  2.86

Total LGBM training time: 1612s


## 8. Blend Weight Optimization

Find the optimal CatBoost weight `w` such that `blended = w * CatBoost + (1-w) * LGBM`
minimizes the average validation Winkler score across all targets.

In [8]:
w_candidates = [0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
blend_weights = {}  # {region: optimal_w_catboost}

alpha_width = 1.0 - (q_hi - q_lo)

for region in REGIONS:
    print(f"\n{'='*55}")
    print(f"Blend weight search: {region}")
    print(f"{'='*55}")
    df_vl = df_val_all[region]

    best_w, best_mean_ws = 1.0, np.inf

    for w in w_candidates:
        ws_list = []
        for tgt in speed_targets:
            if tgt not in cb_speed_models[region]:
                continue
            mask_vl = df_vl[tgt].notna()
            feats = selected_features_all[region][tgt]
            X_vl = df_vl.loc[mask_vl, feats].fillna(0)
            y_vl = df_vl.loc[mask_vl, tgt].values

            # CatBoost predictions (multi-seed average)
            seed_models = cb_speed_models[region][tgt]
            cb_q05 = np.mean([np.maximum(sm[q_lo].predict(X_vl), 0)
                              for sm in seed_models], axis=0)
            cb_q95 = np.mean([sm[q_hi].predict(X_vl)
                              for sm in seed_models], axis=0)

            # LGBM predictions
            lgbm_m = lgbm_blend_models[region][tgt]
            lg_q05 = np.maximum(lgbm_m[q_lo].predict(X_vl), 0)
            lg_q95 = lgbm_m[q_hi].predict(X_vl)

            # Blend
            blend_q05 = w * cb_q05 + (1 - w) * lg_q05
            blend_q95 = w * cb_q95 + (1 - w) * lg_q95
            ws = winkler_score(y_vl, blend_q05, blend_q95, alpha=alpha_width)
            ws_list.append(ws)

        mean_ws = np.mean(ws_list)
        print(f"  w_cb={w:.1f}: mean WS={mean_ws:.3f}")
        if mean_ws < best_mean_ws:
            best_mean_ws = mean_ws
            best_w = w

    blend_weights[region] = best_w
    print(f"  -> Best: w_cb={best_w:.1f} (WS={best_mean_ws:.3f})")

print(f"\nOptimal blend weights: {blend_weights}")


Blend weight search: north_sea


  w_cb=0.4: mean WS=10.734


  w_cb=0.5: mean WS=10.854


  w_cb=0.6: mean WS=11.003


  w_cb=0.7: mean WS=11.184


...

  w_cb=0.5: mean WS=8.714


  w_cb=0.6: mean WS=8.782


  w_cb=0.7: mean WS=8.867


  w_cb=0.8: mean WS=8.967


  w_cb=1.0: mean WS=9.218
  -> Best: w_cb=0.4 (WS=8.659)

Optimal blend weights: {'north_sea': 0.4, 'east_china_sea': 0.4}


## 9. Direction Models -- LGBM

Top-25 features per target, using all features.

In [9]:
# Direction Models — LGBM sin/cos
# Self-contained: loads data directly, no dependency on other cells' variables
import lightgbm as lgb
from utils import load_train_data, exclude_worldwide_features, circular_mae
import time as _time

dir_models = {}
t0 = _time.time()

for region in REGIONS:
    df_dir, feature_cols_dir, _, dir_targets_list = load_train_data(FEATURES_DIR, region)
    df_train_dir = df_dir[df_dir["time"].dt.year.isin([2019, 2020])]
    
    dir_models[region] = {}
    for tgt in dir_targets_list:
        mask_tr = df_train_dir[tgt].notna()
        if mask_tr.sum() < 100:
            continue
        
        # Feature selection (top-25)
        X_sub = df_train_dir.loc[mask_tr, feature_cols_dir].fillna(0).sample(
            n=min(100_000, mask_tr.sum()), random_state=42)
        y_sub = df_train_dir.loc[X_sub.index, tgt]
        m_imp = lgb.LGBMRegressor(n_estimators=50, max_depth=4, verbose=-1, n_jobs=-1)
        m_imp.fit(X_sub, np.sin(np.radians(y_sub)))
        dir_feats = pd.Series(m_imp.feature_importances_, index=feature_cols_dir).nlargest(25).index.tolist()
        
        X_tr = df_train_dir.loc[mask_tr, dir_feats].fillna(0)
        y_tr_sin = np.sin(np.radians(df_train_dir.loc[mask_tr, tgt]))
        y_tr_cos = np.cos(np.radians(df_train_dir.loc[mask_tr, tgt]))
        
        params = dict(n_estimators=200, max_depth=7, learning_rate=0.05,
                      subsample=0.8, verbose=-1, n_jobs=-1)
        m_sin = lgb.LGBMRegressor(**params)
        m_cos = lgb.LGBMRegressor(**params)
        m_sin.fit(X_tr, y_tr_sin)
        m_cos.fit(X_tr, y_tr_cos)
        dir_models[region][tgt] = (m_sin, m_cos, dir_feats)
    
    print(f"  {region}: {len(dir_models[region])} direction models trained")
    del df_dir, df_train_dir

print(f"Direction training complete in {_time.time() - t0:.0f}s")


  north_sea: 12 direction models trained


  east_china_sea: 12 direction models trained
Direction training complete in 240s


In [10]:
dir_offsets = {}  # fallback if computation fails
try:
    # Compute direction prediction intervals
    from utils import compute_direction_intervals
    
    dir_offsets = {}
    for region in REGIONS:
        df_tr_dir, fcols_dir, _, dir_tgts = load_train_data(FEATURES_DIR, region)
        df_tr_dir = df_tr_dir[df_tr_dir["time"].dt.year.isin([2019, 2020])]
        dir_offsets[region] = compute_direction_intervals(
            df_tr_dir, dir_tgts, fcols_dir, dir_models[region], quantile_hi=0.975)
        print(f"  {region}: " + ", ".join(f"+{h}d: +/-{v:.1f} deg" for h, v in dir_offsets[region].items()))
        del df_tr_dir
    
except Exception as e:
    print(f"Direction interval computation failed: {e}")
    print("Submission will not have dir_05/dir_95 columns")


  north_sea: +1d: +/-67.7 deg, +7d: +/-135.8 deg, +14d: +/-136.4 deg


  east_china_sea: +1d: +/-92.9 deg, +7d: +/-143.2 deg, +14d: +/-143.6 deg


## 10. Evaluation Summary

Compare CatBoost-only, LGBM-only, and blended performance across all horizons and
both regions. The blend should match or beat each individual model.

In [11]:
alpha_width = 1.0 - (q_hi - q_lo)

for region in REGIONS:
    print(f"\n{'='*70}")
    print(f"  EVALUATION: {region}  (blend weight: w_cb={blend_weights[region]:.1f})")
    print(f"{'='*70}")
    df_vl = df_val_all[region]
    w = blend_weights[region]

    eval_rows = []
    for tgt in speed_targets:
        if tgt not in cb_speed_models[region]:
            continue
        mask_vl = df_vl[tgt].notna()
        feats = selected_features_all[region][tgt]
        X_vl = df_vl.loc[mask_vl, feats].fillna(0)
        y_vl = df_vl.loc[mask_vl, tgt].values

        horizon = int(tgt.split("_")[1][1:])
        hour = int(tgt.split("_")[2][1:])

        # CatBoost (multi-seed average)
        seed_models = cb_speed_models[region][tgt]
        cb_q05 = np.mean([np.maximum(sm[q_lo].predict(X_vl), 0)
                          for sm in seed_models], axis=0)
        cb_q50 = np.mean([sm[q_mid].predict(X_vl) for sm in seed_models], axis=0)
        cb_q95 = np.mean([sm[q_hi].predict(X_vl) for sm in seed_models], axis=0)

        # LGBM
        lgbm_m = lgbm_blend_models[region][tgt]
        lg_q05 = np.maximum(lgbm_m[q_lo].predict(X_vl), 0)
        lg_q50 = lgbm_m[q_mid].predict(X_vl)
        lg_q95 = lgbm_m[q_hi].predict(X_vl)

        # Blend
        bl_q05 = w * cb_q05 + (1 - w) * lg_q05
        bl_q50 = w * cb_q50 + (1 - w) * lg_q50
        bl_q95 = w * cb_q95 + (1 - w) * lg_q95

        eval_rows.append({
            "target": tgt, "horizon": horizon, "hour": hour,
            "cb_ws": winkler_score(y_vl, cb_q05, cb_q95, alpha=alpha_width),
            "cb_rmse": np.sqrt(np.nanmean((y_vl - cb_q50) ** 2)),
            "lg_ws": winkler_score(y_vl, lg_q05, lg_q95, alpha=alpha_width),
            "lg_rmse": np.sqrt(np.nanmean((y_vl - lg_q50) ** 2)),
            "bl_ws": winkler_score(y_vl, bl_q05, bl_q95, alpha=alpha_width),
            "bl_rmse": np.sqrt(np.nanmean((y_vl - bl_q50) ** 2)),
        })

    eval_df = pd.DataFrame(eval_rows)

    # Summary by horizon
    summary = eval_df.groupby("horizon").agg({
        "cb_ws": "mean", "cb_rmse": "mean",
        "lg_ws": "mean", "lg_rmse": "mean",
        "bl_ws": "mean", "bl_rmse": "mean",
    }).round(2)
    summary.columns = ["CB_WS", "CB_RMSE", "LG_WS", "LG_RMSE", "Blend_WS", "Blend_RMSE"]
    print("\nSpeed (Winkler / RMSE by horizon):")
    print(summary)

    # Overall
    print(f"\nOverall mean WS: CatBoost={eval_df['cb_ws'].mean():.2f}, "
          f"LGBM={eval_df['lg_ws'].mean():.2f}, "
          f"Blend={eval_df['bl_ws'].mean():.2f}")

    # Direction
    if False:  # dir eval skipped
        dir_df = {}
        print(f"\nDirection cMAE by horizon:")
        print(dir_df.groupby("horizon")["cmae"].mean().round(1))
        print(f"Overall mean cMAE: {dir_df['cmae'].mean():.1f} deg")



  EVALUATION: north_sea  (blend weight: w_cb=0.4)



Speed (Winkler / RMSE by horizon):
         CB_WS  CB_RMSE  LG_WS  LG_RMSE  Blend_WS  Blend_RMSE
horizon                                                      
1         5.56     1.25   5.60     1.24      5.56        1.24
7        14.92     3.22  12.58     3.12     13.02        3.14
14       15.33     3.47  13.35     3.41     13.62        3.42

Overall mean WS: CatBoost=11.94, LGBM=10.51, Blend=10.73

  EVALUATION: east_china_sea  (blend weight: w_cb=0.4)



Speed (Winkler / RMSE by horizon):
         CB_WS  CB_RMSE  LG_WS  LG_RMSE  Blend_WS  Blend_RMSE
horizon                                                      
1         4.38     1.03   4.41     1.02      4.37        1.02
7        10.10     2.27   9.47     2.25      9.55        2.25
14       13.18     2.89  11.81     2.86     12.05        2.86

Overall mean WS: CatBoost=9.22, LGBM=8.56, Blend=8.66


## 11. Generate Submission

In [12]:
# Load vertical ratios for level expansion
vertical_ratios = {}
for region in REGIONS:
    vr = load_vertical_ratios(FEATURES_DIR, region)
    vertical_ratios[region] = vr
    if vr is not None:
        levels = vr["level"].unique()
        print(f"{region}: loaded vertical ratios ({len(levels)} levels: {sorted(levels)})")
    else:
        print(f"{region}: no vertical ratios found, will use power-law fallback")

north_sea: loaded vertical ratios (6 levels: ['1000', '100m', '500', '700', '850', '925'])
east_china_sea: loaded vertical ratios (6 levels: ['1000', '100m', '500', '700', '850', '925'])


In [13]:
print("Generating submission with CatBoost+LGBM blend...")
print(f"  Quantiles: {QUANTILES}")
print(f"  Blend weights (CatBoost): {blend_weights}")
t0 = time()

submission = generate_submission(
    speed_models=cb_speed_models,
    dir_models=dir_models,
    selected_features=selected_features_all,
    feature_cols_all=feature_cols_all,
    vertical_ratios=vertical_ratios,
    features_dir=FEATURES_DIR,
    regions=REGIONS,
    n_windows=8,
    quantiles=QUANTILES,
    dir_offsets=dir_offsets,
    blend_models=lgbm_blend_models,
    blend_weights=blend_weights,
)

OUT_PATH = Path("predictions_medium.csv")
submission.to_csv(OUT_PATH, index=False)
elapsed = time() - t0
# --- Package for Codabench upload ---
# Codabench expects a ZIP containing a file named exactly "predictions.csv".
import zipfile
submission_zip = Path("submission_medium.zip")
with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT_PATH, arcname="predictions.csv")
print(f"Ready to upload to Codabench: {submission_zip}")


print(f"\nSubmission saved: {OUT_PATH}")
print(f"  Rows: {len(submission):,}")
print(f"  Regions: {sorted(submission['region'].unique())}")
print(f"  Windows: {sorted(submission['window'].unique())}")
print(f"  Levels: {sorted(submission['level'].unique())}")
print(f"  Generation time: {elapsed:.0f}s")
print(f"\nRows per level:")
print(submission["level"].value_counts().sort_index())
print(f"\nSample rows:")
submission.head(10)


Generating submission with CatBoost+LGBM blend...
  Quantiles: [0.04, 0.5, 0.96]
  Blend weights (CatBoost): {'north_sea': 0.4, 'east_china_sea': 0.4}


  W1 north_sea: 215,460 (7 levels)


  W1 east_china_sea: 215,460 (7 levels)


  W2 north_sea: 215,460 (7 levels)


  W2 east_china_sea: 215,460 (7 levels)


...

  W7 east_china_sea: 215,460 (7 levels)


  W8 north_sea: 215,460 (7 levels)


  W8 east_china_sea: 215,460 (7 levels)



Submission saved: predictions_medium.csv
  Rows: 3,447,360
  Regions: ['east_china_sea', 'north_sea']
  Windows: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
  Levels: ['1000', '100m', '10m', '500', '700', '850', '925']
  Generation time: 83s

Rows per level:
level
1000    492480
100m    492480
10m     492480
500     492480
700     492480
850     492480
925     492480
Name: count, dtype: int64

Sample rows:


,window,region,latitude,longitude,horizon,hour,level,q05,q50,q95,dir_50,dir_05,dir_95
0,1,north_sea,51.0,-4.00,14,0,10m,1.26556,4.05622,8.05838,220.5,84.1,356.9
1,1,north_sea,51.0,-3.75,14,0,10m,1.27566,4.02842,8.10008,220.5,84.1,356.9
2,1,north_sea,51.0,-3.50,14,0,10m,1.28180,3.92912,7.97378,221.4,85.0,357.8
3,1,north_sea,51.0,-3.25,14,0,10m,1.24558,3.87412,7.71460,222.5,86.1,358.9
4,1,north_sea,51.0,-3.00,14,0,10m,1.27512,3.88178,7.59158,222.5,86.1,358.9
5,1,north_sea,51.0,-2.75,14,0,10m,1.31896,3.76908,7.50822,222.5,86.1,358.9
6,1,north_sea,51.0,-2.50,14,0,10m,1.34546,3.85332,7.46042,222.5,86.1,358.9
7,1,north_sea,51.0,-2.25,14,0,10m,1.35310,3.70414,7.42778,222.5,86.1,358.9
8,1,north_sea,51.0,-2.00,14,0,10m,1.39734,3.69216,7.67258,222.5,86.1,358.9
9,1,north_sea,51.0,-1.75,14,0,10m,1.40512,3.73886,7.63088,222.5,86.1,358.9
